In [ ]:
# Client Setup
# This cell configures the external clients used by the notebook:
# - loads environment variables (so API keys can be picked up via .env),
# - initializes the VoyageAI embedding client, and
# - initializes the Anthropic (Claude) client used for chat/reranking.
# We keep the model name in a variable so other cells can reference it.
import voyageai
import json
from dotenv import load_dotenv
from anthropic import Anthropic

# Load environment variables from a .env file in the working directory (if present).
load_dotenv()

# Create API clients. These objects are used later to generate embeddings and
# to call the chat/model API for reranking prompts.
embedding_client = voyageai.Client()
client = Anthropic()
# Model string used when calling the chat API below
model = "claude-haiku-4-5"

In [ ]:
# Helper functions
# This cell defines small helpers used to build chat requests and extract text
# - add_user_message / add_assistant_message: normalize adding messages to the
#   conversation payload accepted by the Anthropic client (role + content).
# - chat: wrapper around client.messages.create to keep call-site code concise.
# - text_from_message: helper to extract plain text blocks from a returned message
#   object (used to parse model outputs in the reranker).
from anthropic.types import Message


def add_user_message(messages, message):
    # Append a user message to the messages list. The incoming 'message' can be
    # an Anthropic Message object or a plain string; this helper normalizes both.
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    # Append an assistant message (same normalization behavior as add_user_message).
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    # Build request params for the Anthropic client. Parameters exposed here mirror
    # the minimal set needed by our prompts (model, messages, temperature, stop sequences).
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    # Call the Anthropic client and return its raw message object. The caller is
    # responsible for extracting text (see text_from_message).
    message = client.messages.create(**params)
    return message


def text_from_message(message):
    # Convert the Anthropic message object into a single text string by joining
    # all text blocks. This keeps downstream parsing simple.
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [ ]:
# Chunk by section
# This cell provides a tiny utility to split a markdown-like document into sections
# using the pattern '\n## ' which commonly denotes a top-level section heading.
import re


def chunk_by_section(document_text):
    # Split the input text on occurrences of a new line followed by '## '.
    # Returns a list of sections (strings). The heading marker is removed by the split
    # so callers may want to handle that if they need the heading text itself.
    pattern = r

    return re.split(pattern, document_text)

In [ ]:
# Embedding Generation
# Wrapper around the VoyageAI embedding client. This function accepts either a
# single string or a list of strings and returns the embedding(s) in the same
# shape as the input (single vector for a string, list of vectors for list input).
def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    # Allow callers to pass a single chunk (string) or many (list). Normalize to list
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    # Call the embedding client. The client returns an object with an 'embeddings'
    # attribute which is a list of vectors matching the input order.
    result = embedding_client.embed(input, model=model, input_type=input_type)
    # Return the embeddings in the same shape as the original input.
    return result.embeddings if is_list else result.embeddings[0]

In [ ]:
# VectorIndex implementation
# This in-memory vector index stores vectors and original document dicts. It supports
# - adding single documents or bulk additions (the bulk path uses the provided
#   embedding function to batch-generate vectors),
# - searching by cosine or euclidean distance, and
# - basic dimension checks and safety validation on inputs.
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        # vectors: list of numeric vectors; documents: corresponding metadata/content
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        # _vector_dim is set on first insertion and used to validate future vectors
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        # embedding_fn: a callable that accepts either a string or list of strings
        # and returns vectors (or list of vectors). Provided by caller (generate_embedding).
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        # Add a single document by computing its embedding and storing vector+doc
        if not self._embedding_fn:
            raise ValueError("Embedding function not provided during initialization.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document dictionary must contain a 'content' key.")

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        # Compute embedding for the single content string and add the vector
        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def add_documents(self, documents: List[Dict[str, Any]]):
        # Bulk-add documents. This minimizes external API calls by batching content
        # into a single embedding client invocation (if embedding_fn supports lists).
        if not self._embedding_fn:
            raise ValueError("Embedding function not provided during initialization.")

        if not isinstance(documents, list):
            raise TypeError("Documents must be a list of dictionaries.")

        if not documents:
            return

        contents = []
        for i, doc in enumerate(documents):
            if not isinstance(doc, dict):
                raise TypeError(f"Document at index {i} must be a dictionary.")
            if "content" not in doc:
                raise ValueError(f"Document at index {i} must contain a 'content' key.")
            if not isinstance(doc["content"], str):
                raise TypeError(f"Document 'content' at index {i} must be a string.")
            contents.append(doc["content"])

        # generate embeddings for all contents in one call
        vectors = self._embedding_fn(contents)

        for vector, document in zip(vectors, documents):
            self.add_vector(vector=vector, document=document)

    def search(self, query: Any, k: int = 1) -> List[Tuple[Dict[str, Any], float]]:
        # Search the stored vectors for the nearest neighbors to 'query'. 'query' can
        # be either a raw string (which will be embedded) or a precomputed vector.
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError("Embedding function not provided for string query.")
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError("Query must be either a string or a list of numbers.")

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        # Choose distance function based on metric
        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        # sort by ascending distance (closer == better)
        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        # Validate input vector and document and then store them in the lists.
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document dictionary must contain a 'content' key.")

        if not self.vectors:
            # first insertion establishes expected vector dimensionality
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        # handle zero-vector edge cases explicitly
        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        # return a distance-like quantity (1 - similarity)
        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"

In [ ]:
# BM25 implementation
from collections import Counter
from typing import Callable, Any, List, Dict, Tuple


class BM25Index:
    def __init__(
        self,
        k1: float = 1.5,
        b: float = 0.75,
        tokenizer: Optional[Callable[[str], List[str]]] = None,
    ):
        self.documents: List[Dict[str, Any]] = []
        self._corpus_tokens: List[List[str]] = []
        self._doc_len: List[int] = []
        self._doc_freqs: Dict[str, int] = {}
        self._avg_doc_len: float = 0.0
        self._idf: Dict[str, float] = {}
        self._index_built: bool = False

        self.k1 = k1
        self.b = b
        self._tokenizer = tokenizer if tokenizer else self._default_tokenizer

    def _default_tokenizer(self, text: str) -> List[str]:
        text = text.lower()
        tokens = re.split(r"\W+", text)
        return [token for token in tokens if token]

    def _update_stats_add(self, doc_tokens: List[str]):
        self._doc_len.append(len(doc_tokens))

        seen_in_doc = set()
        for token in doc_tokens:
            if token not in seen_in_doc:
                self._doc_freqs[token] = self._doc_freqs.get(token, 0) + 1
                seen_in_doc.add(token)

        self._index_built = False

    def _calculate_idf(self):
        N = len(self.documents)
        self._idf = {}
        for term, freq in self._doc_freqs.items():
            idf_score = math.log(((N - freq + 0.5) / (freq + 0.5)) + 1)
            self._idf[term] = idf_score

    def _build_index(self):
        if not self.documents:
            self._avg_doc_len = 0.0
            self._idf = {}
            self._index_built = True
            return

        self._avg_doc_len = sum(self._doc_len) / len(self.documents)
        self._calculate_idf()
        self._index_built = True

    def add_document(self, document: Dict[str, Any]):
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document dictionary must contain a 'content' key.")

        content = document.get("content", "")
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        doc_tokens = self._tokenizer(content)

        self.documents.append(document)
        self._corpus_tokens.append(doc_tokens)
        self._update_stats_add(doc_tokens)

    def add_documents(self, documents: List[Dict[str, Any]]):
        if not isinstance(documents, list):
            raise TypeError("Documents must be a list of dictionaries.")

        if not documents:
            return

        for i, doc in enumerate(documents):
            if not isinstance(doc, dict):
                raise TypeError(f"Document at index {i} must be a dictionary.")
            if "content" not in doc:
                raise ValueError(f"Document at index {i} must contain a 'content' key.")
            if not isinstance(doc["content"], str):
                raise TypeError(f"Document 'content' at index {i} must be a string.")

            content = doc["content"]
            doc_tokens = self._tokenizer(content)

            self.documents.append(doc)
            self._corpus_tokens.append(doc_tokens)
            self._update_stats_add(doc_tokens)

        self._index_built = False

    def _compute_bm25_score(self, query_tokens: List[str], doc_index: int) -> float:
        score = 0.0
        doc_term_counts = Counter(self._corpus_tokens[doc_index])
        doc_length = self._doc_len[doc_index]

        for token in query_tokens:
            if token not in self._idf:
                continue

            idf = self._idf[token]
            term_freq = doc_term_counts.get(token, 0)

            numerator = idf * term_freq * (self.k1 + 1)
            denominator = term_freq + self.k1 * (
                1 - self.b + self.b * (doc_length / self._avg_doc_len)
            )
            score += numerator / (denominator + 1e-9)

        return score

    def search(
        self,
        query: Any,
        k: int = 1,
        score_normalization_factor: float = 0.1,
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.documents:
            return []

        if isinstance(query, str):
            query_text = query
        else:
            raise TypeError("Query must be a string for BM25Index.")

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if not self._index_built:
            self._build_index()

        if self._avg_doc_len == 0:
            return []

        query_tokens = self._tokenizer(query_text)
        if not query_tokens:
            return []

        raw_scores = []
        for i in range(len(self.documents)):
            raw_score = self._compute_bm25_score(query_tokens, i)
            if raw_score > 1e-9:
                raw_scores.append((raw_score, self.documents[i]))

        raw_scores.sort(key=lambda item: item[0], reverse=True)

        normalized_results = []
        for raw_score, doc in raw_scores[:k]:
            normalized_score = math.exp(-score_normalization_factor * raw_score)
            normalized_results.append((doc, normalized_score))

        normalized_results.sort(key=lambda item: item[1])

        return normalized_results

    def __len__(self) -> int:
        return len(self.documents)

    def __repr__(self) -> str:
        return f"BM25VectorStore(count={len(self)}, k1={self.k1}, b={self.b}, index_built={self._index_built})"

In [92]:
# Chunk source text by section
# Read a markdown-like file (report.md) and split it into sections using
# the chunk_by_section helper. This produces a list of text chunks to index.
with open("./report.md", "r") as f:
    text = f.read()

# Each entry in 'chunks' corresponds to a top-level section (split on '\n## ').
chunks = chunk_by_section(text)

In [ ]:
# Reranker function
# This function constructs a single-prompt LLM request to ask the model to select
# and order the top-k most relevant documents for the provided query. The function
# returns a list of document ids (strings) in order of decreasing relevance.
def reranker_fn(docs, query_text, k):
    # Compose a machine-readable representation of the candidate documents so the
    # reranker model can easily parse ids and content. Joining them with newlines keeps
    # the prompt compact but explicit.
    joined_docs = "\n".join(
        [
            f"""
        <document>
        <document_id>{doc["id"]}</document_id>
        <document_content>{doc["content"]}</document_content>
        </document>
        """
            for doc in docs
        ]
    )

    # The prompt instructs the model to return a JSON array of document ids in
    # descending relevance order. Keep prompt clear and include the desired output
    # format so parsing is straightforward.
    prompt = f"""
    You are about to be given a set of documents, along with an id of each.
    Your task is to select and sort the {k} most relevant documents to answer the user's question.

    Here is the user's question:
    <question>
    {query_text}
    </question>
    
    Here are the documents to select from:
    <documents>
    {joined_docs}
    </documents>

    Respond in the following format:
    ```json
    {{
        "document_ids": str[] # List document ids, {k} elements long, sorted in order of decreasing relevance to the user's query. The most relevant documents should be listed first.
    }}
    ```
    """

    # Build the conversation payload for the chat API
    messages = []
    add_user_message(messages, prompt)
    # Hint to the model that we expect JSON by adding a short assistant code fence
    add_assistant_message(messages, "```json")

    # Execute the chat call. We stop at the closing triple-backtick so the returned
    # content contains only the JSON payload we asked for.
    result = chat(messages, stop_sequences=["```"])

    # Parse the JSON returned by the model and extract the ordered document ids.
    # text_from_message converts the model response into a plain string first.
    return json.loads(text_from_message(result))["document_ids"]

In [94]:
# Create a vector index, a bm25 index, then use them to create a Retriever
# Initialize the indexes and wire them into the Retriever. The Retriever will query
# both indexes and optionally call the reranker function to refine ordering.
vector_index = VectorIndex(embedding_fn=generate_embedding)
bm25_index = BM25Index()

retriever = Retriever(bm25_index, vector_index, reranker_fn=reranker_fn)

In [ ]:
# Add all chunks to the retriever, which internally passes them along to both indexes
# Note: we use the bulk 'add_documents' helper to minimize external API calls and
# avoid rate limiting (VoyageAI embedding API may be rate-limited for many small calls).
retriever.add_documents([{"content": chunk} for chunk in chunks])

In [ ]:
# Example search and results printing
# Run a sample query against the retriever and print the top-k results' scores and a
# short preview of their content. Adjust the query and k as needed for experiments.
results = retriever.search("what did the eng team do with INC-2023-Q4-011?", 2)

for doc, score in results:
    # Print the score first and then a 200-character preview of the document content
    print(score, "\n", doc["content"][0:200], "\n---\n")